# mine-tuning-model (RTX 5060 Ti 최적화 버전)

## 사전 준비 (터미널에서 먼저 실행)

```bash
# 1. 가상환경 생성 및 활성화
python -m venv venv
source venv/Scripts/activate

# 2. PyTorch 설치 — RTX 5060 Ti (Blackwell)는 CUDA 12.8 필요!
#    nvidia-smi 실행해서 CUDA 버전 확인 후 아래 명령어 선택

# ✅ CUDA 12.8 (RTX 5060 Ti 권장)
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

# 3. Jupyter 설치
pip install jupyter

# vs code 커널 등록
python -m ipykernel install --user --name venv --display-name "mine-tuning (venv)"

# 4. 노트북 실행
jupyter notebook
```

## 0. GPU 확인 (제일 먼저 실행!)

In [1]:
import subprocess
import sys

# GPU 확인
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
except FileNotFoundError:
    print('⚠️  nvidia-smi를 찾을 수 없습니다. NVIDIA 드라이버가 설치되어 있는지 확인하세요.')

# PyTorch CUDA 확인
import torch
print(f'PyTorch 버전: {torch.__version__}')
print(f'CUDA 버전:    {torch.version.cuda}')
print(f'CUDA 사용 가능: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU 이름: {gpu_name}')
    print(f'GPU VRAM: {vram_gb:.1f}GB')

    # RTX 5060 Ti는 bf16 지원 확인
    bf16_support = torch.cuda.is_bf16_supported()
    print(f'bf16 지원: {bf16_support}')  # True여야 정상
else:
    print('❌ CUDA를 사용할 수 없습니다.')
    print('   → 사전 준비 단계에서 CUDA 12.8 버전 PyTorch를 설치했는지 확인하세요.')
    sys.exit('GPU가 필요합니다.')

Tue May 19 16:16:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.86                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5060 Ti   WDDM  |   00000000:01:00.0 Off |                  N/A |
|  0%   36C    P8              5W /  180W |       0MiB /  16311MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. 필수 라이브러리 설치

> 💡 처음 한 번만 실행하면 돼요!

In [2]:
import subprocess
subprocess.run([
    'pip', 'install',
    'transformers>=4.52.0',
    'trl==0.17.0',
    'datasets',
    'peft',
    'accelerate',
    'bitsandbytes',
    '-q'
], check=True)

print('✅ 패키지 설치 완료')

✅ 패키지 설치 완료


In [ ]:
!pip freeze > requirements-ft.txt

## 2. 데이터 로드 및 Instruction 형식 변환

In [3]:
from datasets import load_dataset

# 데이터 로드
ds = load_dataset('ddorin/minecraft-question-answer-260k')
print(ds)

# 데이터 길이 분포 확인



# Instruction 형식으로 변환
def format_instruction(example):
    return {
        'text': f"""<|im_start|>system
You are an expert Minecraft assistant with deep knowledge of all game mechanics, crafting recipes, biomes, mobs, and strategies.

When answering:
- Be specific and accurate about game mechanics
- Include exact crafting recipes when relevant (materials and placement)
- For survival tips, prioritize safety and efficiency
- Keep answers clear and beginner-friendly unless the question is advanced<|im_end|>
<|im_start|>user
{example['question']}<|im_end|>
<|im_start|>assistant
<think></think>
{example['answer']}<|im_end|>"""
    }

train_data = ds['train'].map(format_instruction, remove_columns=['question', 'answer', 'source'])
print(f'✅ 데이터 변환 완료: {len(train_data)}개')
print('\n샘플 확인:')
print(train_data[0]['text'])

import statistics
lengths = [len(d['text'].split()) for d in train_data]

print(f"평균 길이: {statistics.mean(lengths):.0f} 토큰")
print(f"최대 길이: {max(lengths)} 토큰")
print(f"중앙값:   {statistics.median(lengths):.0f} 토큰")
print(f"75% 이하: {sorted(lengths)[int(len(lengths)*0.75)]} 토큰")
print(f"95% 이하: {sorted(lengths)[int(len(lengths)*0.95)]} 토큰")

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'source'],
        num_rows: 268239
    })
})
✅ 데이터 변환 완료: 268239개

샘플 확인:
<|im_start|>system
You are an expert Minecraft assistant with deep knowledge of all game mechanics, crafting recipes, biomes, mobs, and strategies.

When answering:
- Be specific and accurate about game mechanics
- Include exact crafting recipes when relevant (materials and placement)
- For survival tips, prioritize safety and efficiency
- Keep answers clear and beginner-friendly unless the question is advanced<|im_end|>
<|im_start|>user
What types of leaves can be found naturally in jungle bushes in Minecraft?<|im_end|>
<|im_start|>assistant
<think></think>
In Minecraft, jungle bushes can generate oak leaves in the Java Edition and jungle leaves in the Bedrock Edition.<|im_end|>
평균 길이: 117 토큰
최대 길이: 290 토큰
중앙값:   109 토큰
75% 이하: 133 토큰
95% 이하: 186 토큰


## 3. 모델, 토크나이저 로드

> 💡 4bit 양자화로 로드해서 VRAM을 절약해요. 7B 모델이 약 5~6GB로 줄어들어요.

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import gc

# VRAM 비우기
gc.collect()
torch.cuda.empty_cache()

model_id = 'Qwen/Qwen3.5-9B'

# 4bit 양자화 설정
# RTX 5060 Ti (Blackwell)는 bf16 지원 → compute_dtype을 bf16으로 설정!
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, 
    bnb_4bit_use_double_quant=True,
)

print(f'모델 로딩 중: {model_id}')
print('처음 실행 시 약 18GB 다운로드가 필요해요...')

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={'': 0},  # GPU 0번에 전체 로드
)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

used  = torch.cuda.memory_allocated() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'\n✅ 모델 로드 완료!')
print(f'VRAM 사용: {used:.1f}GB / {total:.1f}GB')

모델 로딩 중: Qwen/Qwen3.5-9B
처음 실행 시 약 18GB 다운로드가 필요해요...


W0519 16:18:08.772000 18972 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

c:\Users\SSAFY\Desktop\ㄷㅇ\mine-tuning-model\venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



✅ 모델 로드 완료!
VRAM 사용: 7.1GB / 15.9GB


## 4. LoRA 어댑터 설정

> 💡 전체 모델을 학습하는 게 아니라 LoRA라는 작은 어댑터만 학습해요.
> 덕분에 VRAM을 훨씬 적게 쓰면서도 파인튜닝 효과를 얻을 수 있어요!

In [5]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4bit 학습 준비
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        'q_proj', 'k_proj',
        'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# 전체 파라미터 중 약 1~2%만 학습된다고 나오면 정상이에요!

trainable params: 29,097,984 || all params: 8,982,901,248 || trainable%: 0.3239


## 5. 학습 실행 (SFTTrainer)

### ⚙️ 매 세션마다 아래 변수만 수정하세요

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
import os

# ============================================================
# 전체 데이터 학습 설정
# ============================================================
from datetime import datetime
date_str = datetime.now().strftime("%Y%m%d")
OUTPUT_DIR = rf"C:\mine-tuning-output\Qwen3.5-9B_{date_str}"

print(f"📦 전체 학습 데이터: {len(train_data):,}건")

# 학습 설정
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    fp16=False,
    logging_steps=50,

    save_strategy="steps",
    save_steps=500,                 # 500스텝마다 중간 저장
    save_total_limit=10,            # 최대 10개 체크포인트 보관

    warmup_steps=100,
    lr_scheduler_type="cosine",
    report_to="none",
    dataloader_num_workers=0,

    optim="paged_adamw_8bit",
    torch_compile=True, 
    max_seq_length=320, 
)

class SaveEpochCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = int(state.epoch)
        save_path = os.path.join(OUTPUT_DIR, f"epoch-{epoch}")
        trainer.model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"✅ 에포크 {epoch} 저장 완료 → {save_path}")

# Trainer 생성
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    args=training_args,
    processing_class=tokenizer,
    callbacks=[SaveEpochCallback()],
)

print("🚀 전체 데이터 학습 시작!")
trainer.train()

# 최종 저장
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\n✅ 전체 학습 완료! → {OUTPUT_DIR}")

📦 전체 학습 데이터: 268,239건


TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'max_seq_length'

## 6. 버전 확인

In [ ]:
import transformers, trl, torch

print(f'transformers: {transformers.__version__}')
print(f'trl:          {trl.__version__}')
print(f'torch:        {torch.__version__}')
print(f'CUDA:         {torch.version.cuda}')

---

## 📋 RTX 5060 Ti 실행 시 주의사항

| 항목 | 내용 |
|------|------|
| **PyTorch** | CUDA 12.8 버전으로 설치 (`--index-url .../cu128`) |
| **bf16** | RTX 5060 Ti (Blackwell)는 `bf16=True` 사용 가능 ✅ |
| **fp16** | bf16 사용 시 반드시 `fp16=False` |
| **bitsandbytes** | `pip install bitsandbytes` (최신 버전 Windows 공식 지원) |
| **dataloader_num_workers** | 반드시 `0`으로 설정 (Windows 멀티프로세싱 이슈) |
| **경로** | `OUTPUT_DIR`에 한글/공백이 없는 경로 사용 권장 |
| **VRAM 부족 시** | `per_device_train_batch_size=2`, `gradient_accumulation_steps=8` |

### 🐛 자주 나오는 에러

```
CUDA error: no kernel image is available for execution on the device
```
→ PyTorch가 5060 Ti를 지원하지 않는 버전이에요. cu128 버전으로 재설치하세요.

```
bitsandbytes: CUDA Setup failed
```
→ `pip install bitsandbytes --upgrade` 로 최신 버전으로 업그레이드하세요.